In [206]:
import xlwings as xw
import os
import tempfile

def capture_sheet_as_image(workbook_path, source_sheet_name, target_path,target_sheet_name,position):

    workbook_path = os.path.abspath(workbook_path)

    workbook_path_dest = os.path.abspath(target_path)    
    wb_target = xw.Book(workbook_path_dest)
    # Verify the workbook exists
    if not os.path.exists(workbook_path):
        print(f"Error: Workbook not found at '{workbook_path}'")
        return False
    
    try:
        # Open the workbook
        wb = xw.Book(workbook_path)
        print(f"Opened workbook: {wb.name}")
        
        # Get source and target sheets
        try:
            source_sheet = wb.sheets[source_sheet_name]
            target_sheet = wb_target.sheets[target_sheet_name]
        except Exception as e:
            print(f"Error finding sheets: {e}")
            return False
        
        # Activate source sheet and get the used range
        source_sheet.activate()
        used_range = source_sheet.used_range
        
        # Method 1: Using clipboard (more reliable)
        try:
            # Copy as picture to clipboard
            used_range.api.CopyPicture()
            
            target_sheet.activate()
            target_sheet.range(position).api.Select()
            target_sheet.api.Paste()
            
            print(f"Successfully copied image from '{source_sheet_name}' to '{target_sheet_name}'")
            wb.save()
            return True
            
        except Exception as e:
            print(f"Error with clipboard method: {e}")
            
            # Method 2: Using temporary file as fallback
            try:
                # Create a temporary file path in the system temp directory
                temp_dir = tempfile.gettempdir()
                temp_image_path = os.path.join(temp_dir, "excel_temp_image.png")
                
                print(f"Trying to save image to: {temp_image_path}")
                
                # Export to image
                source_sheet.api.Export(temp_image_path)
                position = target_sheet.range(position).api.TopLeftCell
                target_sheet.pictures.add(
                    temp_image_path,
                    name='ScreenshotImage',
                    left=0,
                    top= target_sheet.range(f"A{position}").top,
                )
                
                try:
                    os.remove(temp_image_path)
                except:
                    pass
                
                print(f"Successfully captured image from '{source_sheet_name}' to '{target_sheet_name}'")
                wb.save()
                return True
                
            except Exception as e2:
                print(f"Error with temporary file method: {e2}")
                return False
    
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False


In [207]:
import xlwings as xw
import pandas as pd
import re

def split_data_by_poste_with_colors(input_path, sheet_name):

    wb_input = xw.Book(input_path)
    ws_input = wb_input.sheets[sheet_name]
    
    df = pd.read_excel(input_path)
    
    df['POSTE GLOBAL'] = df['POSTE GLOBAL'].apply(lambda x : x.strip().lower().replace(' ',''))
    df['POSTE GLOBAL'] = df['POSTE GLOBAL'].apply(lambda x : re.sub(r'[^A-Za-z0-9 ]', ' ', str(x)))
    unique_postes = df['POSTE GLOBAL'].unique()
    last_sheet = ws_input
    
    for poste in unique_postes:
        
        poste_data = df[df['POSTE GLOBAL'] == poste]
        
        
        ws_output = wb_input.sheets.add(str(poste), after=last_sheet)
        last_sheet = sheet_name_postes
        
        # Write the data to the new sheet
        ws_output.range('A1').value = [df.columns.tolist()] + poste_data.values.tolist()
        

        
        
        # Copy format toooo slowly ........
        # for row_idx, row in enumerate(poste_data.index, start=2):  
        #     for col_idx, _ in enumerate(poste_data.columns, start=1):
        #         input_cell = ws_input.range((row + 1, col_idx))  
        #         output_cell = ws_output.range((row_idx, col_idx))
        #         output_cell.color = input_cell.color  
    
   

In [ ]:
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Side, Border

color_dict = {
    'Y': 'FFFF00',  # Yellow
    'G': '00FF00',  # Green
    'VE': '00FF00',  # Green
    'L': '0000FF',  # Blue
    'R': 'FF0000',  # Red
    'RG': 'FF0000',  # Red
    'W': 'FFFFFF',  # White
    'LG': '808080', # Gray (Light Gray)
    'O': 'FFA500',  # Orange
    'BR': '80471c',#'A52A2A', # Brown
    'MA': '80471c',#'A52A2A', # Brown
    'V': '800080',#'8A2BE2',   # Violet
    'GY': '707070',#'C0C0C0',  # Gray
    'GR': '707070',#'C0C0C0',  # Gray
    'B': '000000' ,  # Black (Noir)
    'NA': '000000' ,  # Black (Noir)
    'P' : 'FFC0CB', # Pink
    'C' : '00FFFF', # Cyan
    'D' : 'FFFFF0', # Ivory
    'SI' : '505050', # Silver
    'BE' : '0000FF',  # Blue
    'JA': 'FFFF00',  # Yellow
    'SA' : 'FFC0CB', # Pink
}

def apply_color(symbol, cell):
    if '/' in str(symbol):
        primary, secondary = symbol.split('/')
        fill = PatternFill(start_color=color_dict[primary], end_color=color_dict[primary], fill_type="solid")
        cell.fill = fill
        
        side = Side(border_style="thick", color=color_dict[secondary])
        border = Border(diagonal=side, diagonalUp=True)
        cell.border = border
    elif symbol in color_dict:
        fill = PatternFill(start_color=color_dict[symbol], end_color=color_dict[symbol], fill_type="solid")
        cell.fill = fill
    else :
        cell.value = symbol
    
    
def schema_final_with_color(wb, color_col):
    unknown_colors = set()
    header_row = 1
    for sheet_name in wb.sheetnames[1:]:
        sheet = wb[sheet_name]
        sheet.insert_cols(color_col+1)
        sheet.cell(row=header_row, column=color_col + 1).value = "Color Formatting"
        for row in range(header_row + 1, sheet.max_row + 1):
            color_symbol = sheet.cell(row=row, column=color_col).value
            if color_symbol:
                cell = sheet.cell(row=row, column=color_col+1)
                try:
                    apply_color(color_symbol, cell)
                except KeyError:
                    print(f"Unknown color code: {color_symbol} in sheet {sheet_name}, row {row}")
                    cell.value = "Unknown color code" #PatternFill(start_color="C0C0C0", end_color="C0C0C0", fill_type="solid")
                    unknown_colors.add(color_symbol)
    print("Color formatting completed successfully.")
    return wb, unknown_colors

In [26]:
path = r"C:\Users\user\Desktop\Connecters\Last Data\Postes.xlsx"
wb = load_workbook(path)

c:\Users\user\anaconda3\envs\rpa_env\Lib\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


In [ ]:
sheet = wb['postes']
for col in range(1, sheet.max_column + 1):
    if sheet.cell(row=1, column=col).value == "Wire Color":
        break

wb, unknown_colors = schema_final_with_color(wb, col)
wb.save(path)

Unknown color code: BE/W in sheet poste7, row 10
Unknown color code: BE/W in sheet lay4, row 9
Unknown color code: BE/W in sheet lay4, row 11
Unknown color code: DL/W in sheet lay4, row 12
Unknown color code: BE/W in sheet lay4, row 13
Unknown color code: DL/W in sheet lay4, row 18
Unknown color code: BE/W in sheet lay4, row 19
Unknown color code: DL/W in sheet lay4, row 22
Unknown color code: DL/BR in sheet lay4, row 30
Unknown color code: BE/W in sheet lay4, row 33
Unknown color code: BE/W in sheet lay4, row 40
Unknown color code: BE/W in sheet lay4, row 43
Unknown color code: BE/BR in sheet hancho, row 4
Unknown color code: BE/W in sheet hancho, row 13
Unknown color code: BE/W in sheet poste10, row 17
Unknown color code: DL/W in sheet poste10, row 19
Unknown color code: BE/P in sheet poste10, row 20
Unknown color code: BE/W in sheet sps1, row 6
Unknown color code: DL/W in sheet sps1, row 10
Unknown color code: BE/W in sheet sps1, row 11
Unknown color code: DL/W in sheet sps1, row 21

In [214]:
def get_sheet_names(wb):
    try:
        sheet_names = [sheet.name for sheet in wb.sheets]
        return sheet_names
        
    except Exception as e:
        print(f"Error getting sheet names: {e}")
        return []
    
def find_first_empty_row(wb, sheet_name, poste):
    image_rows = set()
    ws_input = wb.sheets[poste]
    last_row = ws_input.used_range.last_cell.row
    for pic in ws_input.shapes:
        top_row = 1
        bottom_row = 1
                    
        while ws_input.range(f"A{top_row}").top < pic.top:
            top_row += 1
                    
        bottom_row = top_row
        pic_bottom = pic.top + pic.height
        print(f"bottom row {bottom_row}")
        print(f"pic bottom {pic_bottom}")
        print(f"Last row {last_row}")
        print(ws_input.range(f"A{bottom_row}").top)
        while bottom_row >= last_row and ws_input.range(f"A{bottom_row}").top < pic_bottom:
            
            bottom_row += 1
        print(bottom_row)
        for r in range(top_row-1, bottom_row+1):
            image_rows.add(r)
        print(f"set {image_rows}")
    return max(last_row + 1, max(image_rows) - 1 if image_rows else 0)

In [216]:
def get_connecteurs(liste_connecteurs, chemin_connecteurs, output_file, output_sheet_name):
    chemin_connecteurs = os.path.abspath(chemin_connecteurs)
    output_file = os.path.abspath(output_file)
    
    if not os.path.exists(chemin_connecteurs):
        print(f"Erreur: Fichier connecteurs non trouvé: '{chemin_connecteurs}'")
        return False
        
    if not os.path.exists(output_file):
        print(f"Erreur: Fichier de sortie non trouvé: '{output_file}'")
        return False
    
    
    try :
        wb_connecteurs = xw.Book(chemin_connecteurs)
        wb_output = xw.Book(output_file)
        print(f"Fichiers ouverts: {wb_connecteurs.name} et {wb_output.name}")
        print(get_sheet_names(wb_connecteurs))
        
        for connecteur in liste_connecteurs:
            source_sheet_name = connecteur
            
            if source_sheet_name not in get_sheet_names(wb_connecteurs):
                print(f"Erreur: Feuille '{source_sheet_name}' non trouvée dans '{chemin_connecteurs}'")
                continue
            first_empty_row = find_first_empty_row(wb_output, output_sheet_name, output_sheet_name)
            # find_first_empty_row(wb_output, output_sheet_name)
            position = f"A{first_empty_row}"
            
            capture_sheet_as_image(chemin_connecteurs, source_sheet_name, output_file, output_sheet_name,position)
            print(f"Image de '{source_sheet_name}' capturée dans '{output_sheet_name}' à la position {position}")
            wb_output.save()
        
    except Exception as e:
        print(f"Erreur: {e}")
        return False

In [210]:
def remove_diagonal_line_for_cavity(ws, cavity_number):

    line_name = f"DiagLine_Cavity_{cavity_number}"
    
    for shape in ws.shapes:
        try:
            if shape.name == line_name:
                print(f"Found diagonal line for cavity {cavity_number}, removing it")
                shape.delete()
                return True
        except Exception as e:
            print(f"Error processing shape: {e}")
    
    return False

def remove_fill_from_all_shapes(ws):

    print(f"Processing worksheet: {ws.name}")
    
    for shape in ws.shapes:
        digits = re.match(r'^\d+$', shape.name)
        if digits:
            digits = digits.group(0)
            shape.api.Fill.Visible = False
            remove_diagonal_line_for_cavity(ws, float(shape.name))
            print(f"Shape '{shape.name}': {digits}") 
    else :
        print(f"Shape '{shape.name}': No digits found")

In [ ]:
def color_connector_cavities_corrected(ws):

    print(f"Traitement de la feuille: {ws.name}")
    
    # --- Phase de nettoyage ---
    
    remove_fill_from_all_shapes(ws)
    # --- Phase de traitement ---
    start_row = 16
    current_row = start_row
    
    while ws.range(f"A{current_row}").value is not None and str(ws.range(f"A{current_row}").value).strip() != "":
        try:
            cavity_number = ws.range(f"A{current_row}").value
            cell_b = ws.range(f"B{current_row}")
            
            # Obtenir la couleur de fond
            background_color = cell_b.color
            
            
            if background_color == (0, 0, 0) : 
                print("Noir")
                bg_color_rgb = 0
            elif background_color == (255, 255, 255) :
                bg_color_rgb = 16777215
            else:
                # Conversion standard RGB pour Excel
                r, g, b = background_color
                bg_color_rgb = r + (g << 8) + (b << 16)
            
            # Vérifier si une bordure diagonale montante existe
            has_diagonal = False
            try:
                # 6 = xlDiagonalUp (diagonale montante)
                border_style = cell_b.api.Borders(6).LineStyle
                
                if border_style != -4142:  # -4142 = xlNone
                    line_color = cell_b.api.Borders(6).Color
                    has_diagonal = True
                    print("Diagonal from bottom-left to top-right")
                else:
                    # Si pas de diagonale, ne pas définir de couleur de ligne
                    has_diagonal = False
            except:
                has_diagonal = False
            
            
            if not has_diagonal:
                line_color = 0
                border_style = cell_b.api.Borders(5).LineStyle
                
                if border_style != -4142:  # -4142 = xlNone
                    line_color = cell_b.api.Borders(5).Color
                    has_diagonal = True
                    print("Diagonal from top-left to bottom-right")
                else:
                    # Si pas de diagonale, ne pas définir de couleur de ligne
                    has_diagonal = False
                
            
            found_shape = False
            for shape in ws.shapes:
                try:
                    if shape.api.Type == 6:  
                        continue
                    shape_digits = re.sub(r'[^0-9]', '', shape.name)
                    if shape_digits:
                        shape_digit_str = str(shape_digits)
                        print(shape_digit_str)
                    
                    # if hasattr(shape.api, "TextFrame2") and shape.api.TextFrame2.HasText:
                    #     shape_text = shape.api.TextFrame2.TextRange.Text
                        if shape_digit_str.strip() == str(int(cavity_number)).strip():
                            found_shape = True
                            
                            print(f"Trouvé: Cavité {str(int(cavity_number)).strip()} correspond à forme avec texte '{shape_digit_str.strip()}'")
                            shape.api.Fill.Visible = True
                            shape.api.Fill.Solid()
                            shape.api.Fill.ForeColor.RGB = bg_color_rgb
                            print(f"Couleur de fond: {bg_color_rgb}")
                            
                            # Masquer le contour de la forme
                            shape.api.Line.Visible = False
                            
                            # Dessiner la diagonale seulement si elle existe
                            if has_diagonal:
                                x1 = shape.api.Left
                                y1 = shape.api.Top + shape.api.Height
                                x2 = shape.api.Left + shape.api.Width
                                y2 = shape.api.Top
                                
                                # Ajouter la ligne
                                diag_line = ws.shapes.api.AddLine(x1, y1, x2, y2)
                                
                                # Formater la ligne
                                diag_line.Name = f"DiagLine_Cavity_{cavity_number}"
                                diag_line.Line.ForeColor.RGB = line_color
                                diag_line.Line.Weight = 2.5
                                diag_line.Line.Visible = True
                            
                            break
                except Exception as e:
                    print(f"Erreur avec forme: {e}")
                    continue
            
            if not found_shape:
                print(f"Attention: Forme pour Cavité '{cavity_number}' non trouvée.")
        except Exception as e:
            print(f"Erreur en ligne {current_row}: {e}")
        
        current_row += 1
    
    print("Coloration terminée.")
    return ws

def color_all_Cavities(path):
    wb = xw.Book(path)
    ws = wb.sheets
    for sheet in ws:
        ws = wb.sheets[sheet.name]  
        print(ws)
        color_connector_cavities_corrected(ws)
    wb.save(path)
    

In [30]:
import xlwings as xw
import pandas as pd

path = r"C:\Users\user\Desktop\Connecters\Last Data\Postes.xlsx"
wb = xw.Book(path)
ws = wb.sheets['Postes']

In [ ]:
input_excel_path = r"C:\Users\user\Desktop\Connecters\Last Data\Postes.xlsx"
sheet_name_postes = 'Postes'
split_data_by_poste_with_colors(input_excel_path, sheet_name_postes)

In [193]:
input_excel_path = r"C:\Users\user\Desktop\Connecters\Last Data\Postes.xlsx"
sheet_name_postes = 'Postes'
wb_input = xw.Book(input_excel_path)
ws_input = wb_input.sheets['poste8']

In [199]:
wb_input = xw.Book(input_excel_path)
ws_input = wb_input.sheets['poste8']

In [ ]:
poste_data = df[df['POSTE GLOBAL'] == 'poste8']
list_connecteurs = poste_data['CON-A'].unique()

In [217]:
chemin_connecteurs = r"C:\Users\user\Desktop\Connecters\Copy_c2.xlsx"

for poste in df['POSTE GLOBAL'].unique():
    poste_data = df[df['POSTE GLOBAL'] == poste]
    list_connecteurs = poste_data['CON-A'].unique()
    get_connecteurs(list_connecteurs, chemin_connecteurs, input_excel_path, poste)

Fichiers ouverts: Copy_c2.xlsx et Postes.xlsx
['C72', 'C46', 'C125', 'C140', 'C75', 'C150', 'C59', 'C154', 'C32', 'C53', 'C73', 'C86', 'C64', 'C145', 'C137', 'C160', 'C127', 'C152', 'C171', 'C128', 'C130', 'C69', 'C8', 'C164', 'C133', 'C106', 'C155', 'C92', 'C101', 'C107', 'C83', 'C151', 'C122', 'C33', 'C57', 'C76', 'C78', 'C42', 'C94', 'C109', 'C120', 'C98', 'C96', 'C27', 'C134', 'C102', 'C43', 'C66', 'C5', 'C91', 'C111', 'C13', 'C136', 'C143', 'C15', 'C58', 'C99', 'C20', 'C84', 'C10', 'C9', 'C85', 'C31', 'C4', 'C51', 'C54', 'C44', 'C158', 'C126', 'C123', 'C105', 'C110', 'C114', 'C175', 'C144', 'C97', 'C18', 'C3', 'C40', 'C39', 'C139', 'C16', 'C88', 'C142', 'C138', 'C156', 'C161', 'C104', 'C49', 'C131', 'C135', 'C153', 'C116', 'C172', 'C117', 'C118', 'Sheet2']
Erreur: Feuille 'C28' non trouvée dans 'C:\Users\user\Desktop\Connecters\Copy_c2.xlsx'
Opened workbook: Copy_c2.xlsx
Successfully copied image from 'C16' to 'poste8'
Image de 'C16' capturée dans 'poste8' à la position A54
bottom

In [67]:
import xlwings as xw
import pandas as pd
import os

def display_indices_in_sheets(df, wb, index_groups, poste="poste__None"):

    source_sheet = wb_input.sheets['postes']

    headers = df.columns.tolist()
    
    for i, indices in enumerate(index_groups):
        # Create a new sheet name
        new_sheet_name = f"{poste}"
        
        # Check if sheet exists, if not create it
        if new_sheet_name in [sheet.name for sheet in wb.sheets]:
            target_sheet = wb.sheets[new_sheet_name]
            target_sheet.clear()
        else:
            target_sheet = wb.sheets.add(new_sheet_name, after=wb.sheets[-1])
        
        target_sheet.range('A1').value = headers
        
        print(target_sheet.range(f'A1:{chr(ord("A") + len(headers)-1)}1'))
        header_range = target_sheet.range(f'A1:{chr(ord("A") + len(headers)-1)}1')
        header_range.api.Font.Bold = True
        print(f"Created sheet: {new_sheet_name} with headers")
        # Set header color (example: light blue)
        header_range.color = (173, 216, 230)  # Light blue color
        # For each index in the group, copy the row with formatting
        for row_idx, df_idx in enumerate(indices, start=2):
            # Convert pandas index to Excel row (add 2: 1 for Excel 1-indexing, 1 for header)
            source_excel_row = df_idx + 2
            
            # Copy values
            row_data = df.loc[df_idx].values.tolist()
            target_sheet.range(f'A{row_idx}').value = row_data
            
            # Copy formatting (colors)
            for col_idx in range(len(headers)):
                source_cell = source_sheet.range((source_excel_row, col_idx + 1))
                target_cell = target_sheet.range((row_idx, col_idx + 1))
                
                # Copy cell color
                if source_cell.color:
                    target_cell.color = source_cell.color
        
        # Auto-fit columns for better display
        target_sheet.autofit()
        
    # Save the workbook
    wb.save()
    print(f"Created {len(index_groups)} sheets with the filtered data")


In [71]:
import xlwings as xw
import pandas as pd
import os

def get_column_letter(n):
    """Convert a column number to Excel column letter(s)"""
    result = ""
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        result = chr(65 + remainder) + result
    return result

def display_indices_in_sheets(df, wb, index_groups, poste="poste__None"):
    source_sheet = wb.sheets['postes']  # Fixed capitalization - 'postes' to 'Postes'
    
    headers = df.columns.tolist()
    last_column_letter = get_column_letter(len(headers))
    
    for i, indices in enumerate(index_groups):
        # Create a new sheet name
        new_sheet_name = f"{poste}"
        
        # Check if sheet exists, if not create it
        if new_sheet_name in [sheet.name for sheet in wb.sheets]:
            target_sheet = wb.sheets[new_sheet_name]
            target_sheet.clear()
        else:
            target_sheet = wb.sheets.add(new_sheet_name, after=wb.sheets[-1])
        
        target_sheet.range('A1').value = headers
        
        # Fixed range reference using the column letter function
        header_range = target_sheet.range(f'A1:{last_column_letter}1')
        
        # No need to print the range object itself
        print(f"Header range: A1:{last_column_letter}1")
        
        header_range.api.Font.Bold = True
        print(f"Created sheet: {new_sheet_name} with headers")
        
        # Set header color (light blue)
        header_range.color = (173, 216, 230)
        
        
        for row_idx, df_idx in enumerate(indices, start=2):
            # Convert pandas index to Excel row (add 2: 1 for Excel 1-indexing, 1 for header)
            source_excel_row = df_idx + 2
            
            # Copy values
            row_data = df.loc[df_idx].values.tolist()
            target_sheet.range(f'A{row_idx}').value = row_data
            
            # Copy formatting (colors)
            for col_idx in range(len(headers)):
                source_cell = source_sheet.range((source_excel_row, col_idx + 1))
                target_cell = target_sheet.range((row_idx, col_idx + 1))
                
                # Copy cell color
                if source_cell.color:
                    target_cell.color = source_cell.color
        
        # Auto-fit columns for better display
        target_sheet.autofit()
        
    # Save the workbook
    wb.save()
    print(f"Created {len(index_groups)} sheets with the filtered data")

In [72]:
for poste in unique_postes:
    poste_to_display = df[df['POSTE GLOBAL'] == poste]
    display_indices_in_sheets(df, wb_input, [poste_to_display.index.tolist()], poste=poste)

Header range: A1:EA1
Created sheet: POSTE8 with headers
Created 1 sheets with the filtered data
Header range: A1:EA1
Created sheet: ATL4 with headers


KeyboardInterrupt: 

In [ ]:
def capture_sheet_as_image(workbook_path, source_sheet_name, target_sheet_name):
    """
    Captures a screenshot of source_sheet and places it in target_sheet
    
    Args:
        workbook_path: Full path to the Excel workbook
        source_sheet_name: Name of the sheet to capture
        target_sheet_name: Name of the sheet where to place the image
    """
    # Ensure the workbook path is absolute
    workbook_path = os.path.abspath(workbook_path)

    workbook_path_dest = os.path.abspath(r"C:\Users\user\Desktop\Connecters\CCopy_c2.xlsx")    
    wb_target = xw.Book(workbook_path_dest)
    # Verify the workbook exists
    if not os.path.exists(workbook_path):
        print(f"Error: Workbook not found at '{workbook_path}'")
        return False
    
    try:
        # Open the workbook
        wb = xw.Book(workbook_path)
        print(f"Opened workbook: {wb.name}")
        
        # Get source and target sheets
        try:
            source_sheet = wb.sheets[source_sheet_name]
            target_sheet = wb_target.sheets[target_sheet_name]
        except Exception as e:
            print(f"Error finding sheets: {e}")
            return False
        
        # Activate source sheet and get the used range
        source_sheet.activate()
        used_range = source_sheet.used_range
        
        max_bottom = 0
        if target_sheet.pictures:
            for pic in target_sheet.pictures:
                pic_bottom = pic.top + pic.height
                max_bottom = max(max_bottom, pic_bottom)
            
            new_row = 1
            while True:
                current_top = target_sheet.range(f"A{new_row}").top
                if current_top > max_bottom:
                    print(f"Found empty row at {new_row}")
                    break
                new_row += 1
        else:
            new_row = 1
        
        # Account for pandas DataFrame index by adding 2 to row position
        new_row += 2
        print(f"Adjusted row position for pandas index: {new_row}")
        
        # Method 1: Using clipboard (more reliable)
        try:
            # Copy as picture to clipboard
            used_range.api.CopyPicture()
            
            # Activate target sheet and paste at adjusted position
            target_sheet.activate()
            target_sheet.range(f'A{new_row}').api.Select()
            target_sheet.api.Paste()
            
            print(f"Successfully copied image from '{source_sheet_name}' to '{target_sheet_name}' at row {new_row}")
            wb.save()
            return True
            
        except Exception as e:
            print(f"Error with clipboard method: {e}")
            
            # Method 2: Using temporary file as fallback
            try:
                # Create a temporary file path in the system temp directory
                temp_dir = tempfile.gettempdir()
                temp_image_path = os.path.join(temp_dir, "excel_temp_image.png")
                
                print(f"Trying to save image to: {temp_image_path}")
                
                # Export to image
                source_sheet.api.Export(temp_image_path)
                
                # Add image to target sheet at adjusted position
                target_sheet.pictures.add(
                    temp_image_path,
                    name='ScreenshotImage',
                    left=0,
                    top=target_sheet.range(f"A{new_row}").top,
                )
                
                # Clean up temporary file
                try:
                    os.remove(temp_image_path)
                except:
                    pass
                
                print(f"Successfully captured image from '{source_sheet_name}' to '{target_sheet_name}' at row {new_row}")
                wb.save()
                return True
                
            except Exception as e2:
                print(f"Error with temporary file method: {e2}")
                return False
    
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False

In [ ]:
def split_data_by_poste_with_colors(input_path, sheet_name):
#     wb_input = xw.Book(input_path)
#     ws_input = wb_input.sheets[sheet_name]
    
#     df = pd.read_excel(input_path)
#     unique_postes = df['POSTE GLOBAL'].unique()
#     last_sheet = ws_input
    
#     for poste in unique_postes:
#         poste_data = df[df['POSTE GLOBAL'] == poste]
#         sheet_name = re.sub(r'[^A-Za-z0-9 ]', ' ', str(poste)).strip()
        
#         # Check if the sheet already exists
#         if sheet_name in [sheet.name for sheet in wb_input.sheets]:
#             # Append a unique identifier to the sheet name
#             counter = 1
#             new_sheet_name = f"{sheet_name}_{counter}"
#             while new_sheet_name in [sheet.name for sheet in wb_input.sheets]:
#                 counter += 1
#                 new_sheet_name = f"{sheet_name}_{counter}"
#             sheet_name = new_sheet_name
        
#         ws_output = wb_input.sheets.add(sheet_name, after=last_sheet)
#         last_sheet = ws_output
        
#         # Write the data to the new sheet
#         ws_output.range('A1').value = [df.columns.tolist()] + poste_data.values.tolist()